In [ ]:
from scipy.io import wavfile
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import scipy as sp
import playsound

samplerate, data = wavfile.read('./piano-c3.wav')

playsound.playsound('piano-c3.wav')

print(f"{samplerate = }")
print(f"{data.shape = }")
print(f"{data.dtype = }")

px.line(y=data[:,0])

In [ ]:
channel0 = data[:,0]
channel1 = data[:,1]


In [ ]:

internal_samplerate = samplerate

def sine_wave(freq, samplecount):
    return np.sin(np.pi*2 * np.linspace(0., freq*samplecount/internal_samplerate, samplecount))
def cosine_wave(freq, samplecount):
    return np.cos(np.pi*2 * np.linspace(0., freq*samplecount/internal_samplerate, samplecount))

freq = 100.
duration = 1
y = channel0.astype('float') / (2**15)
# y = sine_wave(freq, duration*internal_samplerate)
# y = sine_wave(100, 5*internal_samplerate) + 2 * sine_wave(200, 5*internal_samplerate)
x = np.indices((len(y),)).reshape(y.shape)
# dst = sp.fft.dst(y)

print(f"{x.shape = }")
print(f"{y.shape = }")
# print(f"{dst.shape = }")

def bla():
    x = np.linspace(50., 100., 1000)
    px.line(x=x, y=[abs(np.sum(y*sine_wave(freq, len(y)))) for freq in x]).show()
    px.line(x=x, y=[abs(np.sum(y*cosine_wave(freq, len(y)))) for freq in x]).show()

# bla()

def maxfreq(y):
    dst = sp.fft.dst(y)
    am = np.argmax(np.abs(dst))
    derived_freq = (am+1)/len(dst)*internal_samplerate/2
    return derived_freq

def maxfreqs(y, n):
    dst = sp.fft.dst(y)
    ams = np.argsort(dst)[-n:][::-1]
    derived_freqs = (ams+1)/len(dst)*internal_samplerate/2
    return derived_freqs

print(f"{maxfreqs(y, 10) = }")

# am = np.argmax(dst)
# print(f"{am = }")
# print(f"{(am+1)/len(x) = }")
# derived_freq = (am+1)/len(x)*internal_samplerate/2
# derived_freq = maxfreq(y)
# print(f"{derived_freq = }")

# px.line(y=dst[:am*2])

# return

def split(big, small, kernel_size):
    kernel = np.ones(kernel_size)
    a = sp.signal.convolve(big*small, kernel, mode='same')
    s2 = sp.signal.convolve(small*small, kernel, mode='same')
    # a = np.cumsum(big*small)
    # s2 = np.cumsum(small*small)
    # print(s2[:20])
    # s2[s2 < 0.01] = np.inf
    # cumamps = a/s2
    amps = a/s2
    # return big-amps*small, amps*small
    return amps

def split_n(big, n):
    acc = np.zeros(big.shape)
    # freqs = maxfreqs(big, n)
    for i in range(n):
        big2 = (big*big).sum()
        print(f"{big2 = }")
        print(f"{i = }")
        if big2 < 0.1:
            break
        
        freq = maxfreq(big)
        # freq = freqs[i]
        print(f"{freq = }")
        w = sine_wave(freq, len(big))
        amps = split(big, w, 4*int(internal_samplerate/freq))
        # a, b = split(big, sine_wave(freq, len(big)), int(internal_samplerate/freq))
        big -= amps*w
        acc += amps*w

        if i == 0:
            px.line(x=np.linspace(0, len(big)/internal_samplerate, num=len(big)), y=amps).show()

        w = cosine_wave(freq, len(big))
        amps = split(big, w, 4*int(internal_samplerate/freq))
        # a, b = split(big, sine_wave(freq, len(big)), int(internal_samplerate/freq))
        big -= amps*w
        acc += amps*w
        
        if i == 0:
            px.line(x=np.linspace(0, len(big)/internal_samplerate, num=len(big)), y=amps).show()
            
        
    return big, acc

rem, sim = split_n(y, 25)
print(rem)
wavfile.write('bad-apple-small-sines.wav', samplerate, sim)
wavfile.write('bad-apple-small-remaining.wav', samplerate, rem)


#f1 = px.line(y=rem)
#f1.show()
#f11 = px.line(y=sim)
#f11.show()
#f2 = px.line(y=y)
#f2.show()

# y2, p1 = split(y, sine_wave(derived_freq, len(y)))
# print(f"{maxfreq(y2) = }")

# smaller = np.zeros(y.shape)

# a = sine_wave(100., duration)
# b = sine_wave(200., duration)
# px.line(np.cumsum(a*b)/np.cumsum(a*a))
# px.line(y=decomp(sine_wave(100,5*internal_samplerate), 100))

In [ ]:
l=samplerate*5#len(channel0)
second=samplerate

def ts(t):
    return np.linspace(0, t, num=int(t*samplerate))

def freqint1(freq, t):
    return np.sin(2*np.pi * ts(t) * freq)
    return fn

def freqint1c(freq, t):
    return np.cos(2*np.pi * ts(t) * freq)
    return fn

def freqint2(freq_start, freq_end, dt):
    tsr = ts(dt)
    return np.sin(2*np.pi*(tsr * freq_start + (freq_end - freq_start) * tsr * tsr / 2 / dt))
    return fn

amps1 = np.concatenate([
    freqint1(100, 2),
    freqint2(100, 200, 3),
    freqint1(200, 2),
]) * 0.2

amps2 = np.concatenate([
    freqint1(200, 2),
    freqint2(200, 100, 3),
    freqint1(100, 2),
]) * 0.2

amps = (amps1 + amps2) * 0.5
# amps = np.linspace(-0.2, 0.2, num=samplerate)

def triangle(freq, t):
    period = 1/freq
    tsr = ts(t)
    tsr %= period
    sel1 = (0 <= tsr) & (tsr < period/4)
    res = tsr * sel1
    sel2 = (period/4 <= tsr) & (tsr < period/2)
    res += (period/2-tsr) * sel2
    sel3 = (period/2 <= tsr) & (tsr < 3*period/4)
    res += (period/2-tsr) * sel3
    sel4 = (3*period/4 <= tsr) & (tsr < period)
    res += (tsr-period) * sel4
    
    #b = tsr[period/4 <= tsr < period/2]
    #c = tsr[period/2 <= tsr < 3*period/4]
    #d = tsr[3*period/4 <= tsr < period]
    return res/period*4

def weird(freq, t):
    period = 1/freq
    tsr = ts(t)
    tsr %= period
    sel1 = (0 <= tsr) & (tsr < period/2)
    res = (period**2/16-(tsr-period/4)**2) * sel1
    sel2 = (period/2 <= tsr) & (tsr < period)
    res += (-period**2/16+(tsr-3*period/4)**2) * sel2
    return res/period**2*16

# (
# px
#  .line(x=ts(0.1), y=[
#      weird(100, 0.1), 
#      freqint1(100, 0.1),
#      triangle(100, 0.1),
#  ])
#  .show()
# )

# amps = sp.signal.sawtooth(ts(3)*2*np.pi*100)*0.1
# #amps = freqint1(100, 5)
# #amps = triangle(100, 3)
# #amps = weird(100, 5)
# amps = np.concatenate([freqint1(150,1), weird(150,1)])
# amps = np.concatenate([amps, amps, amps])

#amps = (freqint1(100,5) + freqint1(150,5))/2

#px.line(x=ts(5), y=amps).show()

# phase shifted + mixed gives same frequency, different amplitude
# a = freqint1(440,1)
# b = freqint1c(440,1)
# amps = np.concatenate([
#     (a+b)*2**0.5,
#     a+a,
# ])
# amps = np.concatenate([
#     amps, amps, amps,
# ])
# amps *= 0.3

# # amps = (freqint1(440,2) + freqint1c(440,2))*0.3
# px.line(x=ts(1), y=[a, b]).show()
# px.line(x=ts(6), y=amps).show()

def tapered_amp(t, tt):
    ret = np.ones(int(t*samplerate))
    bla = np.linspace(0., 1., num=int(tt*samplerate))
    ret[:len(bla)] -= bla[::-1]
    ret[-len(bla):] -= bla
    return ret

amps = freqint1(440,5) * 0
amps[samplerate:4*samplerate] += freqint1(220,3) * tapered_amp(3, 0.05)
amps *= 0.1

single_beat = np.concatenate([
    ts(0.225)*0,
    freqint1(128,0.05) * tapered_amp(0.05,0.01),
    ts(0.225)*0,
])

def beat(t):
    return np.concatenate([single_beat]*t*2)

amps += beat(5)*0.5

px.line(x=ts(5), y=amps).show()

wavfile.write("bla.wav", samplerate, amps)
playsound.playsound("bla.wav")

In [ ]:
a,b = wavfile.read("bla.wav")
print(a)
print(b)